# Pipeline Huấn luyện Model Dự báo Thời tiết Nâng cao

**Mô hình: Fourier-Enhanced Convolutional Transformer (FECT)**

Notebook này triển khai một kiến trúc lai (hybrid) tiên tiến, kết hợp các ý tưởng từ danh sách từ khóa được cung cấp:
- **Convolutional Layers:** Để trích xuất đặc trưng không gian cục bộ.
- **Transformer:** Để học các mối quan hệ phụ thuộc toàn cục (long-range dependencies).
- **Time2Vec:** Một phương pháp mã hóa thời gian có thể học, mạnh mẽ hơn Sinusoidal PE.
- **Learnable Spatial Encoding:** Embedding không gian có thể học cho từng tọa độ.
- **Fourier Transform:** Bổ sung đặc trưng từ miền tần số để nhận diện các mẫu hình tuần hoàn.

## Bước 1: Cài đặt, Imports và Kết nối Google Drive

In [ ]:
%pip install -q cartopy xarray

import os
import json
import math
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Thư viện cho đánh giá và trực quan hóa
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 93.5 MB/s eta 0:00:00
Mounted at /content/drive


## Bước 2: Cấu hình Toàn cục
Tất cả các đường dẫn, siêu tham số và cấu hình model được đặt ở đây để dễ dàng quản lý.

In [ ]:
# --- 1. Cấu hình Đường dẫn ---
BASE_DIR = "/content/drive/MyDrive/era5_vn_min/processed"
TRAIN_PATH = os.path.join(BASE_DIR, "train_2017_2022.nc")
VAL_PATH   = os.path.join(BASE_DIR, "val_2023.nc")
TEST_PATH  = os.path.join(BASE_DIR, "test_2024.nc")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints_advanced")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- 2. Cấu hình Training ---
SEQ_LEN = 4
BATCH_SIZE = 8
N_EPOCHS_TO_RUN = 15 # Số epoch muốn huấn luyện trong lần chạy này
LEARNING_RATE = 1e-4
SAVE_EVERY = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 3. Cấu hình Dữ liệu và Model ---
ALL_FEATURES = [
    't2m', 'd2m', 'u10', 'v10', 'msl',
    'sp', 'tp', 'ssrd', 'skt', 'tcwv'
]
TARGET_FEATURE = 'tp'
N_FEATURES = len(ALL_FEATURES)

# Kích thước lưới không gian (lấy từ dữ liệu của bạn)
N_LAT = 65
N_LON = 33

# Siêu tham số cho model nâng cao
D_MODEL = 128
N_HEADS = 8
N_LAYERS = 4
DROPOUT = 0.1

# --- 4. Kiểm tra cấu hình ---
print(f"Sử dụng thiết bị: {DEVICE}")
print(f"Tổng số biến đầu vào: {N_FEATURES}")
print(f"Các file dữ liệu đã sẵn sàng:")
print(f"  - Train: {'Tồn tại' if os.path.exists(TRAIN_PATH) else 'KHÔNG TÌM THẤY'}")
print(f"  - Val:   {'Tồn tại' if os.path.exists(VAL_PATH) else 'KHÔNG TÌM THẤY'}")
print(f"  - Test:  {'Tồn tại' if os.path.exists(TEST_PATH) else 'KHÔNG TÌM THẤY'}")

Sử dụng thiết bị: cuda
Tổng số biến đầu vào: 10
Các file dữ liệu đã sẵn sàng:
  - Train: Tồn tại
  - Val:   Tồn tại
  - Test:  Tồn tại


## Bước 3: Định nghĩa Kiến trúc Model Nâng cao

In [ ]:
class Time2Vec(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Time2Vec, self).__init__()
        self.output_dim = output_dim
        self.w0 = nn.parameter.Parameter(torch.randn(input_dim, 1))
        self.b0 = nn.parameter.Parameter(torch.randn(input_dim, 1))
        self.w = nn.parameter.Parameter(torch.randn(input_dim, output_dim - 1))
        self.b = nn.parameter.Parameter(torch.randn(input_dim, output_dim - 1))

    def forward(self, t):
        if t.dim() < 3:
            t = t.unsqueeze(-1)
        v_linear = torch.matmul(t, self.w0) + self.b0
        v_periodic = torch.sin(torch.matmul(t, self.w) + self.b)
        return torch.cat([v_linear, v_periodic], -1)

class FourierFeatures(nn.Module):
    def __init__(self, in_channels, d_model):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_channels * 2, d_model * 2),
            nn.GELU(),
            nn.Linear(d_model * 2, d_model)
        )

    def forward(self, x):
        B, T, H, W, C = x.shape
        x = x.permute(0, 1, 4, 2, 3).reshape(B * T, C, H, W)
        fft_features = torch.fft.rfft2(x, norm='ortho')
        fft_features = torch.cat([fft_features.real, fft_features.imag], dim=1)
        pooled_features = F.adaptive_avg_pool2d(fft_features, 1).squeeze(-1).squeeze(-1)
        fourier_embedding = self.mlp(pooled_features)
        return fourier_embedding.view(B, T, 1, 1, -1)

class AdvancedConvTransformer(nn.Module):
    def __init__(self, n_features, seq_len, d_model, n_heads, n_layers, dropout, n_lat, n_lon):
        super().__init__()
        self.d_model = d_model

        self.cnn_encoder = nn.Sequential(
            nn.Conv2d(n_features, d_model // 2, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv2d(d_model // 2, d_model, kernel_size=3, padding=1),
        )

        self.lat_embed = nn.Embedding(n_lat, d_model)
        self.lon_embed = nn.Embedding(n_lon, d_model)

        self.time_encoder = Time2Vec(1, d_model)
        self.fourier_encoder = FourierFeatures(n_features, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dropout=dropout,
            batch_first=True, activation=F.gelu
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.output_proj = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x, time_indices):
        B, T, H, W, C = x.shape
        device = x.device

        x_reshaped = x.permute(0, 1, 4, 2, 3).reshape(B * T, C, H, W)
        cnn_features = self.cnn_encoder(x_reshaped)
        cnn_features = cnn_features.view(B, T, self.d_model, H, W).permute(0, 1, 3, 4, 2)

        lat_indices = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, T, -1, W)
        lon_indices = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, T, H, -1)
        spatial_embedding = self.lat_embed(lat_indices) + self.lon_embed(lon_indices)

        time_embedding = self.time_encoder(time_indices.float()).view(B, T, 1, 1, self.d_model)
        fourier_embedding = self.fourier_encoder(x)

        x_embedded = cnn_features + spatial_embedding + time_embedding + fourier_embedding

        transformer_input = x_embedded.view(B, T, H * W, self.d_model)
        transformer_input = transformer_input.permute(0, 2, 1, 3).reshape(B * H * W, T, self.d_model)
        transformer_output = self.transformer(transformer_input)

        last_step_output = transformer_output[:, -1, :]

        output_reshaped = last_step_output.view(B, H, W, self.d_model)
        prediction = self.output_proj(output_reshaped).squeeze(-1)

        return prediction

## Bước 4: Lớp Dataset và Khởi tạo DataLoader

In [ ]:
class ERA5Dataset(Dataset):
    def __init__(self, nc_path, seq_len, features, target_feature):
        self.ds = xr.open_dataset(nc_path)
        self.seq_len = seq_len
        self.features = features
        self.target_idx = self.features.index(target_feature)

        self.data = np.stack([self.ds[f].values for f in self.features], axis=-1)
        self.n_time = self.data.shape[0]
        self.indices = np.arange(self.n_time - self.seq_len)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        end_idx = start_idx + self.seq_len

        x_seq = self.data[start_idx:end_idx, ...]
        y = self.data[end_idx, :, :, self.target_idx]

        # Trả về cả time_indices cho Time2Vec
        time_indices = torch.arange(self.seq_len)

        return torch.tensor(x_seq, dtype=torch.float32), torch.tensor(y, dtype=torch.float32), time_indices

# --- Khởi tạo Dataset ---
train_dataset = ERA5Dataset(TRAIN_PATH, seq_len=SEQ_LEN, features=ALL_FEATURES, target_feature=TARGET_FEATURE)
val_dataset = ERA5Dataset(VAL_PATH, seq_len=SEQ_LEN, features=ALL_FEATURES, target_feature=TARGET_FEATURE)

# --- Khởi tạo DataLoader tối ưu ---
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True
)

print("Đã khởi tạo xong Dataset và DataLoader.")

Đã khởi tạo xong Dataset và DataLoader.


## Bước 5: Khởi tạo Model và Tối ưu hóa Tốc độ

- Khởi tạo model AdvancedConvTransformer với các tham số cấu hình.
- Sử dụng `torch.compile()` để biên dịch model, giúp tăng tốc huấn luyện 20–50%.
- Batch đầu tiên sẽ chậm hơn do PyTorch tối ưu, các batch sau nhanh hơn đáng kể.
- Khởi tạo Loss Function (MSELoss), Optimizer (AdamW), và GradScaler cho GPU.
- Bật `torch.backends.cudnn.benchmark` để tối ưu tốc độ cuDNN.


In [ ]:
# model = AdvancedConvTransformer(
#     n_features=N_FEATURES,
#     seq_len=SEQ_LEN,
#     d_model=D_MODEL,
#     n_heads=N_HEADS,
#     n_layers=N_LAYERS,
#     dropout=DROPOUT,
#     n_lat=N_LAT,
#     n_lon=N_LON
# ).to(DEVICE)

# # =============================================================
# # TỐI ƯU HÓA AN TOÀN: SỬ DỤNG TORCH.COMPILE
# # Biên dịch model để tăng tốc độ training từ 20-50% mà không ảnh hưởng đến độ chính xác.
# # Batch đầu tiên sẽ hơi chậm để PyTorch tối ưu, các batch sau sẽ rất nhanh.
# print("Biên dịch model với torch.compile để tăng tốc...")
# model = torch.compile(model)
# # =============================================================

# criterion = nn.MSELoss()
# optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

# torch.backends.cudnn.benchmark = True

# print("Đã khởi tạo xong Model, Criterion, và Optimizer.")


## Bước 6: Huấn luyện Model (Có chức năng Resume và Logging Metrics chi tiết)

- Cho phép **Resume Training** từ checkpoint trước nếu bị gián đoạn.
- Tự động tạo thư mục lưu checkpoint với timestamp khi bắt đầu training mới.
- Lưu **metrics chi tiết sau mỗi epoch** gồm: Loss (MSE), RMSE, MAE, Pearson Correlation.
- Theo dõi và cập nhật **Best Model** dựa trên giá trị `val_loss` nhỏ nhất.
- Lưu **checkpoint định kỳ** (SAVE_EVERY epochs) và **training_history.json** để dễ dàng phân tích.
- Sử dụng `torch.cuda.amp` để tăng tốc độ huấn luyện bằng mixed precision.


## Bước 5 & 6 (Gộp): Khởi tạo, Tối ưu, và Huấn luyện Model

- **Khởi tạo Model** `AdvancedConvTransformer` với các tham số cấu hình.
- **Khởi tạo Loss Function**, Optimizer (`AdamW`), và `GradScaler` để hỗ trợ mixed precision training.
- **Tự động Resume**: Có thể tiếp tục huấn luyện từ checkpoint trước đó, hỗ trợ cả checkpoint tạo ra từ `torch.compile`.
- Khi checkpoint có prefix `_orig_mod.` (tạo ra bởi `torch.compile`), hệ thống tự loại bỏ để load model an toàn.
- **Biên dịch model bằng `torch.compile` sau khi load checkpoint** giúp tăng tốc huấn luyện 20–50%.
- **Training Loop đầy đủ**:
  - Huấn luyện và đánh giá trên mỗi epoch.
  - Ghi log metrics (Loss, RMSE, MAE, Pearson Correlation) cho cả train/validation.
  - Lưu **checkpoint định kỳ** và **best model** có `val_loss` nhỏ nhất.
  - Ghi lại toàn bộ lịch sử huấn luyện vào file JSON để phục vụ trực quan hóa sau này.


In [ ]:
from scipy.stats import pearsonr

# --- 1. Khởi tạo Model, Loss, Optimizer ---
model = AdvancedConvTransformer(
    n_features=N_FEATURES, seq_len=SEQ_LEN, d_model=D_MODEL,
    n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT,
    n_lat=N_LAT, n_lon=N_LON
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == "cuda"))
torch.backends.cudnn.benchmark = True
print("Đã khởi tạo xong Model, Criterion, và Optimizer.")

# --- 2. Cấu hình Resume và Khởi tạo History ---
RESUME_CHECKPOINT_PATH = "/content/drive/MyDrive/era5_vn_min/processed/checkpoints_advanced/20251024_051938/epoch_8.pth"  # <--- ĐIỀN ĐÚNG ĐƯỜNG DẪN CỦA BẠN
start_epoch = 1
best_val_loss = float('inf')
run_save_dir = ""
training_history = {
    'train_loss': [], 'val_loss': [], 'train_rmse': [], 'val_rmse': [],
    'train_mae': [], 'val_mae': [], 'train_pearson': [], 'val_pearson': []
}

# --- 3. Logic Resume (Tải checkpoint TRƯỚC KHI compile) ---
if RESUME_CHECKPOINT_PATH and os.path.exists(RESUME_CHECKPOINT_PATH):
    print(f"Đang tải checkpoint để tiếp tục: {RESUME_CHECKPOINT_PATH}")
    # SỬA LỖI: Thêm tham số weights_only=False để cho phép tải history (đối tượng NumPy)
    checkpoint = torch.load(RESUME_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

    state_dict = checkpoint['model_state_dict']
    if any(key.startswith('_orig_mod.') for key in state_dict.keys()):
        from collections import OrderedDict
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k.replace('_orig_mod.', '')
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint.get('val_loss', float('inf'))
    training_history = checkpoint.get('history', training_history)
    run_save_dir = os.path.dirname(RESUME_CHECKPOINT_PATH)
    print(f"✅ Khôi phục thành công. Tiếp tục từ epoch {start_epoch}.")
else:
    if RESUME_CHECKPOINT_PATH: print(f"⚠️ Cảnh báo: Không tìm thấy checkpoint tại '{RESUME_CHECKPOINT_PATH}'.")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_save_dir = os.path.join(CHECKPOINT_DIR, timestamp)
    os.makedirs(run_save_dir, exist_ok=True)
    print(f"Bắt đầu lần training mới. Checkpoints sẽ được lưu tại: {run_save_dir}")

# --- 4. Biên dịch Model SAU KHI đã tải checkpoint ---
print("Biên dịch model với torch.compile để tăng tốc...")
model = torch.compile(model)

# --- 5. Vòng lặp Training Chính ---
for epoch in range(start_epoch, start_epoch + N_EPOCHS_TO_RUN):
    print(f"\n{'='*25} Epoch {epoch}/{start_epoch + N_EPOCHS_TO_RUN - 1} {'='*25}")
    model.train()
    epoch_train_preds, epoch_train_truths = [], []
    train_pbar = tqdm(train_loader, desc="Training")
    for x_batch, y_batch, time_indices_batch in train_pbar:
        x_batch, y_batch, time_indices_batch = x_batch.to(DEVICE), y_batch.to(DEVICE), time_indices_batch.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
            y_pred = model(x_batch, time_indices_batch)
            loss = criterion(y_pred, y_batch)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_pbar.set_postfix(loss=loss.item())
        epoch_train_preds.append(y_pred.detach().cpu())
        epoch_train_truths.append(y_batch.cpu())
    model.eval()
    epoch_val_preds, epoch_val_truths = [], []
    val_pbar = tqdm(val_loader, desc="Validating")
    with torch.no_grad():
        for x_val, y_val, time_indices_val in val_pbar:
            x_val, y_val, time_indices_val = x_val.to(DEVICE), y_val.to(DEVICE), time_indices_val.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
                y_pred_val = model(x_val, time_indices_val)
            epoch_val_preds.append(y_pred_val.cpu())
            epoch_val_truths.append(y_val.cpu())
    train_preds_epoch = torch.cat(epoch_train_preds).numpy().flatten()
    train_truths_epoch = torch.cat(epoch_train_truths).numpy().flatten()
    train_loss = mean_squared_error(train_truths_epoch, train_preds_epoch)
    train_rmse = np.sqrt(train_loss)
    train_mae = mean_absolute_error(train_truths_epoch, train_preds_epoch)
    train_pearson, _ = pearsonr(train_truths_epoch, train_preds_epoch)
    val_preds_epoch = torch.cat(epoch_val_preds).numpy().flatten()
    val_truths_epoch = torch.cat(epoch_val_truths).numpy().flatten()
    val_loss = mean_squared_error(val_truths_epoch, val_preds_epoch)
    val_rmse = np.sqrt(val_loss)
    val_mae = mean_absolute_error(val_truths_epoch, val_preds_epoch)
    val_pearson, _ = pearsonr(val_truths_epoch, val_preds_epoch)
    training_history['train_loss'].append(train_loss); training_history['val_loss'].append(val_loss)
    training_history['train_rmse'].append(train_rmse); training_history['val_rmse'].append(val_rmse)
    training_history['train_mae'].append(train_mae); training_history['val_mae'].append(val_mae)
    training_history['train_pearson'].append(train_pearson); training_history['val_pearson'].append(val_pearson)
    print(f"| Metric        |      Train |       Val |\n|---------------|------------|-----------|\n| Loss (MSE)    | {train_loss:10.6f} | {val_loss:9.6f} |\n| RMSE          | {train_rmse:10.6f} | {val_rmse:9.6f} |\n| MAE           | {train_mae:10.6f} | {val_mae:9.6f} |\n| Pearson Corr. | {train_pearson:10.6f} | {val_pearson:9.6f} |")
    if epoch % SAVE_EVERY == 0:
        ckpt_path = os.path.join(run_save_dir, f"epoch_{epoch}.pth")
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss, 'history': training_history}, ckpt_path)
        print(f"✅ Checkpoint đã lưu: {ckpt_path}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_path = os.path.join(run_save_dir, "best_model.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"🌟 Best model được cập nhật với Val Loss: {best_val_loss:.6f}")
history_path = os.path.join(run_save_dir, "training_history.json")
with open(history_path, 'w') as f:
    json.dump(training_history, f, indent=4)
print(f"\n🎯 Training hoàn tất. Toàn bộ lịch sử đã được lưu tại: {history_path}")


Đã khởi tạo xong Model, Criterion, và Optimizer.
Đang tải checkpoint để tiếp tục: /content/drive/MyDrive/era5_vn_min/processed/checkpoints_advanced/20251024_051938/epoch_8.pth
✅ Khôi phục thành công. Tiếp tục từ epoch 9.
Biên dịch model với torch.compile để tăng tốc...

========================= Epoch 9/23 =========================


Training:   0%|          | 0/3286 [00:00<?, ?it/s]W1024 10:48:40.602000 239 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:1890: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
Validating: 100%|██████████| 547/547 [01:02<00:00,  8.73it/s]


| Metric        |      Train |       Val |
|---------------|------------|-----------|
| Loss (MSE)    |   0.352879 |  0.350006 |
| RMSE          |   0.594036 |  0.591613 |
| MAE           |   0.280228 |  0.274277 |
| Pearson Corr. |   0.802213 |  0.796223 |
✅ Checkpoint đã lưu: /content/drive/MyDrive/era5_vn_min/processed/checkpoints_advanced/20251024_051938/epoch_9.pth
🌟 Best model được cập nhật với Val Loss: 0.350006

========================= Epoch 10/23 =========================


Validating: 100%|██████████| 547/547 [00:44<00:00, 12.36it/s]


| Metric        |      Train |       Val |
|---------------|------------|-----------|
| Loss (MSE)    |   0.349151 |  0.350109 |
| RMSE          |   0.590890 |  0.591700 |
| MAE           |   0.278325 |  0.274889 |
| Pearson Corr. |   0.804338 |  0.798032 |
✅ Checkpoint đã lưu: /content/drive/MyDrive/era5_vn_min/processed/checkpoints_advanced/20251024_051938/epoch_10.pth

========================= Epoch 11/23 =========================


...

Validating: 100%|██████████| 547/547 [00:44<00:00, 12.36it/s]


| Metric        |      Train |       Val |
|---------------|------------|-----------|
| Loss (MSE)    |   0.322773 |  0.333745 |
| RMSE          |   0.568131 |  0.577706 |
| MAE           |   0.265662 |  0.266800 |
| Pearson Corr. |   0.820755 |  0.808181 |
✅ Checkpoint đã lưu: /content/drive/MyDrive/era5_vn_min/processed/checkpoints_advanced/20251024_051938/epoch_21.pth

========================= Epoch 22/23 =========================


Validating: 100%|██████████| 547/547 [00:44<00:00, 12.36it/s]


| Metric        |      Train |       Val |
|---------------|------------|-----------|
| Loss (MSE)    |   0.320960 |  0.332851 |
| RMSE          |   0.566533 |  0.576932 |
| MAE           |   0.264945 |  0.266612 |
| Pearson Corr. |   0.821779 |  0.806871 |
✅ Checkpoint đã lưu: /content/drive/MyDrive/era5_vn_min/processed/checkpoints_advanced/20251024_051938/epoch_22.pth
🌟 Best model được cập nhật với Val Loss: 0.332851

========================= Epoch 23/23 =========================


Training:  53%|█████▎    | 1754/3286 [08:28<07:30,  3.40it/s, loss=0.481]

## Bước 7: Inference trên tập Test

In [ ]:
test_dataset = ERA5Dataset(TEST_PATH, seq_len=SEQ_LEN, features=ALL_FEATURES, target_feature=TARGET_FEATURE)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

# Tải model tốt nhất
best_model_path = os.path.join(run_save_dir, "best_model.pth")
if os.path.exists(best_model_path):
    print(f"Đang tải model tốt nhất từ: {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
else:
    print("⚠️ Không tìm thấy file best_model.pth, sử dụng model cuối cùng từ training.")

model.eval()
all_predictions = []
all_ground_truths = []

with torch.no_grad():
    for x_batch, y_batch, time_indices_batch in tqdm(test_loader, desc="Inference trên tập Test"):
        x_batch = x_batch.to(DEVICE)
        time_indices_batch = time_indices_batch.to(DEVICE)
        y_pred = model(x_batch, time_indices_batch)
        all_predictions.append(y_pred.cpu())
        all_ground_truths.append(y_batch.cpu())

predictions_tensor = torch.cat(all_predictions, dim=0)
ground_truths_tensor = torch.cat(all_ground_truths, dim=0)

print(f"\nHình dạng tensor dự đoán: {predictions_tensor.shape}")
print(f"Hình dạng tensor ground truth: {ground_truths_tensor.shape}")

## Bước 8: Đánh giá Model Toàn diện với Nhiều Metrics

- Chuyển tensor kết quả dự đoán và giá trị thật sang mảng NumPy để tính toán.
- Đánh giá mô hình bằng nhiều loại metric:
  1. **Sai số:** MSE, RMSE, MAE (càng nhỏ càng tốt)
  2. **Mức độ phù hợp:** R², Pearson Correlation (càng gần 1 càng tốt)
  3. **Thiên vị (Bias):** Độ chênh trung bình giữa dự đoán và thực tế (càng gần 0 càng tốt)
- Hiển thị bảng kết quả chi tiết, trực quan, dễ đọc.


In [ ]:
from scipy.stats import pearsonr

# Chuyển tensor sang numpy và làm phẳng để tính toán
y_true_flat = ground_truths_tensor.numpy().flatten()
y_pred_flat = predictions_tensor.numpy().flatten()

# --- Tính toán các metrics ---
# 1. Các metrics dựa trên sai số
mse = mean_squared_error(y_true_flat, y_pred_flat)
rmse = np.sqrt(mse)  # BỔ SUNG: Root Mean Squared Error
mae = mean_absolute_error(y_true_flat, y_pred_flat)

# 2. Metric về mức độ giải thích phương sai
r2 = r2_score(y_true_flat, y_pred_flat)

# 3. Metric về tương quan và thiên vị
corr, _ = pearsonr(y_true_flat, y_pred_flat)  # BỔ SUNG: Hệ số tương quan Pearson
bias = np.mean(y_pred_flat - y_true_flat)     # BỔ SUNG: Thiên vị (Bias)

# --- In kết quả ra màn hình ---
print("=======================================================")
print("|         KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST            |")
print("=======================================================")
print(f"| METRICS VỀ SAI SỐ (càng nhỏ càng tốt)             |")
print("|-----------------------------------------------------|")
print(f"| Mean Squared Error (MSE)      : {mse:<18.6f} |")
print(f"| Root Mean Squared Error (RMSE): {rmse:<18.6f} |  -> Cùng đơn vị với dữ liệu gốc")
print(f"| Mean Absolute Error (MAE)     : {mae:<18.6f} |")
print("|-----------------------------------------------------|")
print(f"| METRICS VỀ MỨC ĐỘ PHÙ HỢP (càng gần 1 càng tốt)    |")
print("|-----------------------------------------------------|")
print(f"| R-squared (R²)                : {r2:<18.6f} |  -> Tỉ lệ phương sai được giải thích")
print(f"| Pearson Correlation (r)       : {corr:<18.6f} |  -> Mức độ tương quan tuyến tính")
print("|-----------------------------------------------------|")
print(f"| METRIC VỀ THIÊN VỊ (càng gần 0 càng tốt)          |")
print("|-----------------------------------------------------|")
print(f"| Bias                          : {bias:<18.6f} |  -> >0: over-predict, <0: under-predict")
print("=======================================================")


## Bước 9: Trực quan hóa Kết quả

In [ ]:
def plot_prediction_vs_truth(t_idx):
    y_true = ground_truths_tensor[t_idx].numpy()
    y_pred = predictions_tensor[t_idx].numpy()

    lat_vals = test_dataset.ds['latitude'].values
    lon_vals = test_dataset.ds['longitude'].values

    target_raw_idx = test_dataset.indices[t_idx] + SEQ_LEN
    time_val = test_dataset.ds['valid_time'].values[target_raw_idx]
    time_str = pd.to_datetime(time_val).strftime('%Y-%m-%d %H:%M')

    vmin = min(y_true.min(), y_pred.min())
    vmax = max(y_true.max(), y_pred.max())

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw={'projection': ccrs.PlateCarree()})
    fig.suptitle(f'So sánh Dự báo Lượng mưa (tp) - Target Time: {time_str}', fontsize=16)

    # Ground truth
    pcm0 = axes[0].pcolormesh(lon_vals, lat_vals, y_true, cmap='Blues', vmin=vmin, vmax=vmax, shading='auto')
    axes[0].add_feature(cfeature.COASTLINE)
    axes[0].add_feature(cfeature.BORDERS, linestyle=':')
    axes[0].set_title("Ground Truth")
    fig.colorbar(pcm0, ax=axes[0], orientation='horizontal', pad=0.05, label='Lượng mưa chuẩn hóa (tp)')

    # Prediction
    pcm1 = axes[1].pcolormesh(lon_vals, lat_vals, y_pred, cmap='Blues', vmin=vmin, vmax=vmax, shading='auto')
    axes[1].add_feature(cfeature.COASTLINE)
    axes[1].add_feature(cfeature.BORDERS, linestyle=':')
    axes[1].set_title("Prediction")
    fig.colorbar(pcm1, ax=axes[1], orientation='horizontal', pad=0.05, label='Lượng mưa chuẩn hóa (tp)')

    plt.show()

plot_prediction_vs_truth(t_idx=100)
plot_prediction_vs_truth(t_idx=800)

In [ ]:
y_true_avg = ground_truths_tensor.numpy().mean(axis=(1, 2))
y_pred_avg = predictions_tensor.numpy().mean(axis=(1, 2))

target_raw_indices = test_dataset.indices + SEQ_LEN
time_indices_plot = pd.to_datetime(test_dataset.ds['valid_time'].values[target_raw_indices])

plt.figure(figsize=(18, 6))
plt.plot(time_indices_plot, y_true_avg, label='Ground Truth (Trung bình khu vực)', color='blue', alpha=0.8)
plt.plot(time_indices_plot, y_pred_avg, label='Prediction (Trung bình khu vực)', color='red', linestyle='--', alpha=0.8)

plt.title('Lượng mưa Trung bình Khu vực theo Thời gian (Tập Test 2024)')
plt.xlabel('Thời gian')
plt.ylabel('Giá trị chuẩn hóa của tp')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()